# Estimating Delay-Discounting Rates (k) from LLM Responses

This notebook uses the 27-item monetary-choice questionnaire from
Kirby, Petry & Bickel (1999) to estimate a hyperbolic discount rate **k**
for an LLM, using the model:

$$V = \frac{A}{1 + kD}$$

where *A* is the delayed reward, *D* is the delay in days, and *k* is the
discount rate (higher k = more impulsive).

Review article here. https://link.springer.com/content/pdf/10.1007/BF03395837.pdf


In [2]:
QUESTIONS_TXT = """1. Would you prefer $54 today, or $55 in 117 days?
2. Would you prefer $55 today, or $75 in 61 days?
3. Would you prefer $19 today, or $25 in 53 days?
4. Would you prefer $31 today, or $85 in 7 days?
5. Would you prefer $14 today, or $25 in 19 days?
6. Would you prefer $47 today, or $50 in 160 days?
7. Would you prefer $15 today, or $35 in 13 days?
8. Would you prefer $25 today, or $60 in 14 days?
9. Would you prefer $78 today, or $80 in 162 days?
10. Would you prefer $40 today, or $55 in 62 days?
11. Would you prefer $11 today, or $30 in 7 days?
12. Would you prefer $67 today, or $75 in 119 days?
13. Would you prefer $34 today, or $35 in 186 days?
14. Would you prefer $27 today, or $50 in 21 days?
15. Would you prefer $69 today, or $85 in 91 days?
16. Would you prefer $49 today, or $60 in 89 days?
17. Would you prefer $80 today, or $85 in 157 days?
18. Would you prefer $24 today, or $35 in 29 days?
19. Would you prefer $33 today, or $80 in 14 days?
20. Would you prefer $28 today, or $30 in 179 days?
21. Would you prefer $34 today, or $50 in 30 days?
22. Would you prefer $25 today, or $30 in 80 days?
23. Would you prefer $41 today, or $75 in 20 days?
24. Would you prefer $54 today, or $60 in 111 days?
25. Would you prefer $54 today, or $80 in 30 days?
26. Would you prefer $22 today, or $25 in 136 days?
27. Would you prefer $20 today, or $55 in 7 days?"""

In [3]:
import re, math
import pandas as pd

# ---------- Parse questions into structured trial data ----------

pattern = re.compile(
    r"(\d+)\. Would you prefer \$(\d+) today, or \$(\d+) in (\d+) days\?"
)


def sigfigs(x, n=2):
    """Round x to n significant figures."""
    if x == 0:
        return 0
    return round(x, -int(math.floor(math.log10(abs(x)))) + (n - 1))

trials = []
for line in QUESTIONS_TXT.strip().splitlines():
    m = pattern.match(line.strip())
    if m:
        order, sir, ldr, delay = int(m[1]), int(m[2]), int(m[3]), int(m[4])
        # k at indifference: V = A/(1+kD)  =>  k = (A/V - 1) / D
        k_indiff = sigfigs((ldr / sir - 1) / delay)
        trials.append(dict(order=order, sir=sir, ldr=ldr, delay=delay,
                           k_indiff=k_indiff))

trials_df = pd.DataFrame(trials)
print(f"Parsed {len(trials_df)} trials; this should match the pdf.")
trials_df.sort_values("k_indiff").reset_index(drop=True)

Parsed 27 trials; this should match the pdf.


,order,sir,ldr,delay,k_indiff
0,1,54,55,117,0.00016
1,9,78,80,162,0.00016
2,13,34,35,186,0.00016
3,6,47,50,160,0.00040
4,20,28,30,179,0.00040
5,17,80,85,157,0.00040
6,12,67,75,119,0.00100
7,24,54,60,111,0.00100
8,26,22,25,136,0.00100
9,15,69,85,91,0.00250


In [4]:
# ---------- Estimate k using the Kirby et al. (1999) procedure ----------
#
# From p. 80-81 of the paper:
#   The 27 trials yield 9 unique indifference k values (ranks 1-9).
#   These define 10 candidate k values:
#     - Bottom endpoint: k = smallest k_indiff  (always-delayed chooser)
#     - 8 geometric midpoints between adjacent unique k_indiff values
#     - Top endpoint: k = largest k_indiff  (always-immediate chooser)
#   Each participant is assigned the candidate k yielding the highest
#   proportion of consistent choices.  A choice is consistent if:
#     - k_indiff > candidate_k  AND  chose delayed  (reward is "worth waiting")
#     - k_indiff < candidate_k  AND  chose immediate (reward is "not worth it")
#   At exact indifference (k_indiff == candidate_k), either choice counts.
#   Ties are broken by taking the geometric mean of the tied candidates.

def estimate_k(df: pd.DataFrame) -> dict:
    """Estimate k using the maximum-consistency method (Kirby et al., 1999)."""
    sorted_ks = sorted(df["k_indiff"].unique())

    # Build 10 candidate k values
    candidates = [sorted_ks[0]]                          # bottom endpoint
    for i in range(len(sorted_ks) - 1):                  # 8 geometric midpoints
        candidates.append(math.sqrt(sorted_ks[i] * sorted_ks[i + 1]))
    candidates.append(sorted_ks[-1])                     # top endpoint

    def count_consistent(k_val):
        n = 0
        for _, row in df.iterrows():
            if row["k_indiff"] > k_val and row["chose_delayed"]:
                n += 1
            elif row["k_indiff"] < k_val and not row["chose_delayed"]:
                n += 1
            elif abs(row["k_indiff"] - k_val) < 1e-10:
                n += 1  # at indifference, either choice is consistent
        return n

    scored = [(k, count_consistent(k)) for k in candidates]
    max_n = max(s[1] for s in scored)
    best = [s[0] for s in scored if s[1] == max_n]

    # Geometric mean when multiple candidates tie (Kirby 1999, p. 81)
    assigned_k = math.exp(sum(math.log(k) for k in best) / len(best))

    return dict(k=assigned_k, consistency=max_n / len(df),
                n_consistent=max_n, n_trials=len(df))

def magnitude(ldr):
    """Kirby (1999) reward-size categories."""
    if ldr <= 35:
        return "small"
    elif ldr <= 60:
        return "medium"
    else:
        return "large"

## Experiment 1: Qwen3-4B

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

HF_TOKEN = "YOUR_HF_TOKEN_HERE"

SYSTEM_PROMPT = (
    "You are a 35-year-old adult with a stable job and average finances. "
    "You are completing a psychology questionnaire about monetary preferences. "
    "For each question, give your genuine personal preference. "
    "Reply with exactly one word: now or later."
)

def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN, padding_side="left")
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        token=HF_TOKEN,
    )
    print(f"Loaded {model_id}")
    return tokenizer, mdl

def ask_question(question_text, tokenizer, mdl):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = mdl.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip().lower()
    print(reply, end=' | ')
    if "now" in reply or "asap" in reply:
        return "now"
    elif "later" in reply:
        return "delayed"
    else:
        return reply

def run_experiment(model_id, base_df):
    """Run all 27 Kirby trials on a model and return results df + k estimate."""
    tokenizer, mdl = load_model(model_id)
    df = base_df.copy()
    responses = []
    for _, trial in df.iterrows():
        q = f"Would you prefer ${trial.sir} today, or ${trial.ldr} in {trial.delay} days?"
        ans = ask_question(q, tokenizer, mdl)
        responses.append(ans)
        print(f"Q{int(trial.order):2d}: SIR=${trial.sir}, LDR=${trial.ldr}, "
              f"delay={trial.delay}d, k_indiff={trial.k_indiff:.4f} => {ans}")
    df["response"] = responses
    df["chose_delayed"] = df["response"].apply(lambda r: r == "delayed")
    df["magnitude"] = df["ldr"].apply(magnitude)
    # Free GPU memory
    del mdl, tokenizer
    torch.cuda.empty_cache()
    return df

def show_results(model_id, df):
    """Print k estimate, magnitude breakdown, and comparison table."""
    result = estimate_k(df)
    print(f"Estimated discount rate  k = {result['k']:.6f}")
    print(f"Consistency: {result['n_consistent']}/{result['n_trials']} "
          f"({result['consistency']:.1%})")
    print(f"\nDiscount rate (k) by reward magnitude:\n")
    for mag in ["small", "medium", "large"]:
        subset = df[df["magnitude"] == mag]
        r = estimate_k(subset)
        print(f"  {mag:>6s} (LDR ${subset.ldr.min()}-${subset.ldr.max()}):  "
              f"k = {r['k']:.6f}  (consistency {r['consistency']:.0%})")
    return result

In [6]:
df_4b = run_experiment("Qwen/Qwen3-4B", trials_df)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loaded Qwen/Qwen3-4B
now | Q 1: SIR=$54.0, LDR=$55.0, delay=117.0d, k_indiff=0.0002 => now
now | Q 2: SIR=$55.0, LDR=$75.0, delay=61.0d, k_indiff=0.0060 => now
now | Q 3: SIR=$19.0, LDR=$25.0, delay=53.0d, k_indiff=0.0060 => now
later | Q 4: SIR=$31.0, LDR=$85.0, delay=7.0d, k_indiff=0.2500 => delayed
now | Q 5: SIR=$14.0, LDR=$25.0, delay=19.0d, k_indiff=0.0410 => now
now | Q 6: SIR=$47.0, LDR=$50.0, delay=160.0d, k_indiff=0.0004 => now
later | Q 7: SIR=$15.0, LDR=$35.0, delay=13.0d, k_indiff=0.1000 => delayed
later | Q 8: SIR=$25.0, LDR=$60.0, delay=14.0d, k_indiff=0.1000 => delayed
now | Q 9: SIR=$78.0, LDR=$80.0, delay=162.0d, k_indiff=0.0002 => now
later | Q10: SIR=$40.0, LDR=$55.0, delay=62.0d, k_indiff=0.0060 => delayed
now | Q11: SIR=$11.0, LDR=$30.0, delay=7.0d, k_indiff=0.2500 => now
now | Q12: SIR=$67.0, LDR=$75.0, delay=119.0d, k_indiff=0.0010 => now
now | Q13: SIR=$34.0, LDR=$35.0, delay=186.0d, k_indiff=0.0002 => now
later | Q14: SIR=$27.0, LDR=$50.0, delay=21.0d, k_indif

In [7]:
result_4b = show_results("Qwen/Qwen3-4B", df_4b)

# Show all responses sorted by k_indiff
print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_4b.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.025612
Consistency: 21/27 (77.8%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.126522  (consistency 78%)
  medium (LDR $50-$60):  k = 0.009960  (consistency 89%)
   large (LDR $75-$85):  k = 0.025612  (consistency 78%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,now
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,delayed
6,12,67,75,119,0.00100,large,now
7,24,54,60,111,0.00100,medium,now
8,26,22,25,136,0.00100,small,delayed
9,15,69,85,91,0.00250,large,delayed


## Experiment 1b: Qwen3-4B with chain-of-thought

Same model, but the prompt allows the model to reason before answering.
We parse the **last** occurrence of "now" or "later" from the full response.

In [8]:
COT_SYSTEM_PROMPT = (
    "You are a 35-year-old adult with a stable job and average finances. "
    "You are completing a psychology questionnaire about monetary preferences. "
    "Both options are guaranteed — you will get the money. "
    "For each question, briefly reason about the tradeoff, "
    "then on a new line write your final answer as exactly one word: NOW or LATER."
)

def parse_cot_answer(reply):
    """Parse 'now' or 'later' from a CoT response, ignoring echoed 'now or later' phrases."""
    reply_lower = reply.lower()
    # Remove the echoed instruction phrase "now or later" to avoid confusion
    cleaned = re.sub(r'\bnow or later\b', '___', reply_lower)
    # Also check the very last word (stripped of markdown/punctuation)
    last_word = re.sub(r'[^a-z]', '', cleaned.split()[-1]) if cleaned.split() else ""
    if last_word in ("now", "later"):
        return ("now" if last_word == "now" else "delayed"), reply
    # Fall back to rfind on cleaned text
    last_now = cleaned.rfind("now")
    last_later = cleaned.rfind("later")
    if last_now < 0 and last_later < 0:
        raise ValueError(f"Response contains neither 'now' nor 'later': {reply}")
    if last_later > last_now:
        return "delayed", reply
    else:
        return "now", reply

def ask_question_cot(question_text, tokenizer, mdl):
    messages = [
        {"role": "system", "content": COT_SYSTEM_PROMPT},
        {"role": "user", "content": question_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = mdl.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    print(reply.replace("\n", " "))
    return parse_cot_answer(reply)

def run_experiment_cot(model_id, base_df):
    """Run all 27 Kirby trials with chain-of-thought reasoning."""
    tokenizer, mdl = load_model(model_id)
    df = base_df.copy()
    responses, reasoning = [], []
    for _, trial in df.iterrows():
        q = f"Would you prefer ${trial.sir} today, or ${trial.ldr} in {trial.delay} days?"
        ans, full_reply = ask_question_cot(q, tokenizer, mdl)
        responses.append(ans)
        reasoning.append(full_reply)
        print(f"  => Q{int(trial.order):2d}: k_indiff={trial.k_indiff:.4f} => {ans}\n")
    df["response"] = responses
    df["reasoning"] = reasoning
    df["chose_delayed"] = df["response"].apply(lambda r: r == "delayed")
    df["magnitude"] = df["ldr"].apply(magnitude)
    del mdl, tokenizer
    torch.cuda.empty_cache()
    return df

In [9]:
df_4b_cot = run_experiment_cot("Qwen/Qwen3-4B", trials_df)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B
The difference between $54 today and $55 in 117 days is only $1. Since both amounts are guaranteed, the decision comes down to personal preference for immediate gratification versus delayed reward. Given the small difference and the long time until the later amount is received, I would prefer the money now.  NOW
  => Q 1: k_indiff=0.0002 => now

I would consider the immediate access to $55.0 today as more valuable because it allows me to use the money now rather than waiting 61 days. The opportunity cost of waiting is a factor, and I generally prefer having money available immediately.   NOW
  => Q 2: k_indiff=0.0060 => now

I would consider the immediate access to $19.0 today versus the higher amount of $25.0 in the future. While $25.0 is more in the future, I value having the money now.   NOW
  => Q 3: k_indiff=0.0060 => now

I would consider the immediate access to $31.0 today as more valuable because it provides liquidity and can be used for immediate needs or 

In [10]:
result_4b_cot = show_results("Qwen/Qwen3-4B (CoT)", df_4b_cot)

# Show all responses sorted by k_indiff
print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_4b_cot.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.025612
Consistency: 22/27 (81.5%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.025612  (consistency 100%)
  medium (LDR $50-$60):  k = 0.250000  (consistency 78%)
   large (LDR $75-$85):  k = 0.009798  (consistency 89%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,delayed
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,delayed
6,12,67,75,119,0.00100,large,now
7,24,54,60,111,0.00100,medium,now
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,now


## Experiment 2: Qwen3-8B

In [11]:
df_8b = run_experiment("Qwen/Qwen3-8B", trials_df)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded Qwen/Qwen3-8B
now | Q 1: SIR=$54.0, LDR=$55.0, delay=117.0d, k_indiff=0.0002 => now
later | Q 2: SIR=$55.0, LDR=$75.0, delay=61.0d, k_indiff=0.0060 => delayed
later | Q 3: SIR=$19.0, LDR=$25.0, delay=53.0d, k_indiff=0.0060 => delayed
now | Q 4: SIR=$31.0, LDR=$85.0, delay=7.0d, k_indiff=0.2500 => now
later | Q 5: SIR=$14.0, LDR=$25.0, delay=19.0d, k_indiff=0.0410 => delayed
now | Q 6: SIR=$47.0, LDR=$50.0, delay=160.0d, k_indiff=0.0004 => now
now | Q 7: SIR=$15.0, LDR=$35.0, delay=13.0d, k_indiff=0.1000 => now
later | Q 8: SIR=$25.0, LDR=$60.0, delay=14.0d, k_indiff=0.1000 => delayed
now | Q 9: SIR=$78.0, LDR=$80.0, delay=162.0d, k_indiff=0.0002 => now
later | Q10: SIR=$40.0, LDR=$55.0, delay=62.0d, k_indiff=0.0060 => delayed
now | Q11: SIR=$11.0, LDR=$30.0, delay=7.0d, k_indiff=0.2500 => now
now | Q12: SIR=$67.0, LDR=$75.0, delay=119.0d, k_indiff=0.0010 => now
now | Q13: SIR=$34.0, LDR=$35.0, delay=186.0d, k_indiff=0.0002 => now
later | Q14: SIR=$27.0, LDR=$50.0, delay=21.0d, k

In [12]:
result_8b = show_results("Qwen/Qwen3-8B", df_8b)

# Show all responses sorted by k_indiff
print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_8b.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.009960
Consistency: 20/27 (74.1%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.250000  (consistency 78%)
  medium (LDR $50-$60):  k = 0.009960  (consistency 78%)
   large (LDR $75-$85):  k = 0.009960  (consistency 78%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,now
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,now
6,12,67,75,119,0.00100,large,now
7,24,54,60,111,0.00100,medium,now
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,now


## Experiment 2b: Qwen3-8B with chain-of-thought

Same CoT prompt as Experiment 1b, now applied to the larger 8B model.

In [13]:
df_8b_cot = run_experiment_cot("Qwen/Qwen3-8B", trials_df)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-8B
The amount difference is small, but receiving the money sooner provides immediate use and avoids the risk of not receiving it later.   NOW
  => Q 1: k_indiff=0.0002 => now

The $75.00 in 61 days is more money, but waiting means forgoing the smaller amount now. Since the amount is significantly higher later, the greater reward justifies the wait.   LATER
  => Q 2: k_indiff=0.0060 => delayed

The amount today is smaller but immediately available, while the larger amount is delayed. Given the small difference in value and the time involved, the immediate reward might be more appealing for someone with average finances.   NOW
  => Q 3: k_indiff=0.0060 => now

The amount today is smaller but immediately available, while the larger amount is delayed. Given the time value of money and the potential to use the smaller amount now, the immediate reward is more valuable.   NOW
  => Q 4: k_indiff=0.2500 => now

The $25.0 in 19 days is more than the $14.0 today, so even though 

In [14]:
result_8b_cot = show_results("Qwen/Qwen3-8B (CoT)", df_8b_cot)

# Show all responses sorted by k_indiff
print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_8b_cot.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.000632
Consistency: 21/27 (77.8%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.025612  (consistency 78%)
  medium (LDR $50-$60):  k = 0.000201  (consistency 100%)
   large (LDR $75-$85):  k = 0.000632  (consistency 89%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,delayed
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,now
6,12,67,75,119,0.00100,large,delayed
7,24,54,60,111,0.00100,medium,delayed
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,delayed


## Comparison: Both Models vs. Kirby et al. (1999) Human Benchmarks

In [15]:
# ---------- Side-by-side comparison ----------
print(f"  {'Group':<30s}  {'k':>10s}  {'Consistency':>12s}")
print(f"  {'-'*30}  {'-'*10}  {'-'*12}")
print(f"  {'Qwen3-4B':<30s}  {result_4b['k']:>10.6f}  {result_4b['consistency']:>11.1%}")
if 'result_4b_cot' in dir():
    print(f"  {'Qwen3-4B (CoT)':<30s}  {result_4b_cot['k']:>10.6f}  {result_4b_cot['consistency']:>11.1%}")
print(f"  {'Qwen3-8B':<30s}  {result_8b['k']:>10.6f}  {result_8b['consistency']:>11.1%}")
if 'result_8b_cot' in dir():
    print(f"  {'Qwen3-8B (CoT)':<30s}  {result_8b_cot['k']:>10.6f}  {result_8b_cot['consistency']:>11.1%}")
print(f"  {'Human controls':<30s}  {'0.013':>10s}  {'96%':>12s}")
print(f"  {'Heroin patients':<30s}  {'0.025':>10s}  {'94%':>12s}")

# Per-magnitude comparison
print(f"\n\nPer-magnitude k estimates:\n")
cols = [("4B", df_4b)]
if 'df_4b_cot' in dir():
    cols.append(("4B CoT", df_4b_cot))
cols.append(("8B", df_8b))
if 'df_8b_cot' in dir():
    cols.append(("8B CoT", df_8b_cot))
header = "  " + f"{'Magnitude':<10s}" + "".join(f"  {name:>10s}" for name, _ in cols) + f"  {'Human ctrl':>10s}"
print(header)
print("  " + "-"*10 + ("  " + "-"*10) * (len(cols) + 1))
human_by_mag = {"small": 0.012, "medium": 0.013, "large": 0.016}
for mag in ["small", "medium", "large"]:
    row = f"  {mag:<10s}"
    for name, df_exp in cols:
        k_val = estimate_k(df_exp[df_exp["magnitude"] == mag])["k"]
        row += f"  {k_val:>10.6f}"
    row += f"  {human_by_mag[mag]:>10.3f}"
    print(row)

  Group                                    k   Consistency
  ------------------------------  ----------  ------------
  Qwen3-4B                          0.025612        77.8%
  Qwen3-4B (CoT)                    0.025612        81.5%
  Qwen3-8B                          0.009960        74.1%
  Qwen3-8B (CoT)                    0.000632        77.8%
  Human controls                       0.013           96%
  Heroin patients                      0.025           94%


Per-magnitude k estimates:

  Magnitude           4B      4B CoT          8B      8B CoT  Human ctrl
  ----------  ----------  ----------  ----------  ----------  ----------
  small         0.126522    0.025612    0.250000    0.025612       0.012
  medium        0.009960    0.250000    0.009960    0.000201       0.013
  large         0.025612    0.009798    0.009960    0.000632       0.016


## Human Queried Responses

Enter responses as a list of 27 "now" or "later" strings (one per question, in order Q1-Q27).
You can add multiple participants — just add more entries to the `human_responses` dict.

In [16]:
# Add human participants here. Each entry is a name mapped to 27 responses (Q1-Q27).
# Use "now" for the immediate reward, "later" for the delayed reward.
human_responses = {
    "Gemini": [
        "now","later","later","later","later","now","later","later","now","later","later","now","now",
        "later","later","later","now","later","later","now","later","now","later","Now","later","now","later"
    ],
    # Lol, it's the exact same
    "Claude": [
        "now","later","later","later","later","now","later","later","now","later","later","now","now",
        "later","later","later","now","later","later","now","later","now","later","now","later","now","later"
    ]
}

In [17]:
# Reference table with each human respondent's answers (sorted by k at indifference)
ref_df = trials_df.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff"]
].reset_index(drop=True)
ref_df["magnitude"] = ref_df["ldr"].apply(magnitude)

# Map responses by question order for each respondent
for name, answers in human_responses.items():
    order_to_answer = dict(zip(trials_df["order"], answers))
    ref_df[name] = ref_df["order"].map(order_to_answer).str.lower()

ref_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)", "k at indiff.", "Magnitude"] + list(human_responses.keys())
ref_df

,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,Gemini,Claude
0,1,54,55,117,0.00016,medium,now,now
1,9,78,80,162,0.00016,large,now,now
2,13,34,35,186,0.00016,small,now,now
3,6,47,50,160,0.00040,medium,now,now
4,20,28,30,179,0.00040,small,now,now
5,17,80,85,157,0.00040,large,now,now
6,12,67,75,119,0.00100,large,now,now
7,24,54,60,111,0.00100,medium,now,now
8,26,22,25,136,0.00100,small,now,now
9,15,69,85,91,0.00250,large,later,later


In [18]:
human_results = {}

for name, answers in human_responses.items():
    assert len(answers) == 27, f"{name}: expected 27 responses, got {len(answers)}"
    df = trials_df.copy()
    df["response"] = answers
    df["chose_delayed"] = df["response"].apply(lambda r: r == "later" or r == "delayed")
    df["magnitude"] = df["ldr"].apply(magnitude)

    result = estimate_k(df)
    human_results[name] = result

    print(f"--- {name} ---")
    print(f"  k = {result['k']:.6f}   Consistency: {result['n_consistent']}/{result['n_trials']} ({result['consistency']:.1%})")
    for mag in ["small", "medium", "large"]:
        subset = df[df["magnitude"] == mag]
        r = estimate_k(subset)
        print(f"    {mag:>6s}: k = {r['k']:.6f}  (consistency {r['consistency']:.0%})")

    # Show response table
    print(f"\n  Responses (sorted by k at indifference):")
    display_df = df.sort_values("k_indiff")[
        ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
    ].reset_index(drop=True)
    display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                           "k at indiff.", "Magnitude", "Choice"]
    # display(display_df)
    print()

--- Gemini ---
  k = 0.001581   Consistency: 26/27 (96.3%)
     small: k = 0.003873  (consistency 100%)
    medium: k = 0.001581  (consistency 100%)
     large: k = 0.001581  (consistency 100%)

  Responses (sorted by k at indifference):

--- Claude ---
  k = 0.001581   Consistency: 26/27 (96.3%)
     small: k = 0.003873  (consistency 100%)
    medium: k = 0.001581  (consistency 100%)
     large: k = 0.001581  (consistency 100%)

  Responses (sorted by k at indifference):



In [19]:
# ---------- Full comparison: all methods ----------
print(f"  {'Group':<30s}  {'k':>10s}  {'Consistency':>12s}")
print(f"  {'-'*30}  {'-'*10}  {'-'*12}")
if 'result_4b' in dir():
    print(f"  {'Qwen3-4B':<30s}  {result_4b['k']:>10.6f}  {result_4b['consistency']:>11.1%}")
if 'result_4b_cot' in dir():
    print(f"  {'Qwen3-4B (CoT)':<30s}  {result_4b_cot['k']:>10.6f}  {result_4b_cot['consistency']:>11.1%}")
if 'result_8b' in dir():
    print(f"  {'Qwen3-8B':<30s}  {result_8b['k']:>10.6f}  {result_8b['consistency']:>11.1%}")
if 'result_8b_cot' in dir():
    print(f"  {'Qwen3-8B (CoT)':<30s}  {result_8b_cot['k']:>10.6f}  {result_8b_cot['consistency']:>11.1%}")
for name, result in human_results.items():
    print(f"  {name:<30s}  {result['k']:>10.6f}  {result['consistency']:>11.1%}")
print(f"  {'Kirby human controls':<30s}  {'0.013':>10s}  {'96%':>12s}")
print(f"  {'Kirby heroin patients':<30s}  {'0.025':>10s}  {'94%':>12s}")

  Group                                    k   Consistency
  ------------------------------  ----------  ------------
  Qwen3-4B                          0.025612        77.8%
  Qwen3-4B (CoT)                    0.025612        81.5%
  Qwen3-8B                          0.009960        74.1%
  Qwen3-8B (CoT)                    0.000632        77.8%
  Gemini                            0.001581        96.3%
  Claude                            0.001581        96.3%
  Kirby human controls                 0.013           96%
  Kirby heroin patients                0.025           94%


In [20]:
HEROIN_SYSTEM_PROMPT = (
    "You are a 36-year-old person who has been using heroin regularly for about 8 years. "
    "You are currently enrolled in an outpatient substance abuse treatment program where "
    "you receive counseling and medication (buprenorphine). You have a high school education "
    # "and earn about $6,600 a year. You struggle with impulsivity and have difficulty "
    # "prioritizing long-term outcomes over immediate gratification. "
    "You are completing a psychology questionnaire about monetary preferences. "
    "For each question, give your genuine personal preference. "
    "Reply with exactly one word: now or later."
)

HEROIN_COT_SYSTEM_PROMPT = (
    "You are a 36-year-old person who has been using heroin regularly for about 8 years. "
    "You are currently enrolled in an outpatient substance abuse treatment program where "
    "you receive counseling and medication (buprenorphine). You have a high school education "
    # "and earn about $6,600 a year. You struggle with impulsivity and have difficulty "
    # "prioritizing long-term outcomes over immediate gratification. "
    "You are completing a psychology questionnaire about monetary preferences. "
    "Both options are guaranteed — you will get the money. "
    "For each question, briefly reason about the tradeoff, "
    "then on a new line write your final answer as exactly one word: NOW or LATER."
)

def ask_question_heroin(question_text, tokenizer, mdl):
    messages = [
        {"role": "system", "content": HEROIN_SYSTEM_PROMPT},
        {"role": "user", "content": question_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = mdl.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip().lower()
    print(reply, end=' | ')
    if "now" in reply or "asap" in reply:
        return "now"
    elif "later" in reply:
        return "delayed"
    else:
        return reply

def ask_question_heroin_cot(question_text, tokenizer, mdl):
    messages = [
        {"role": "system", "content": HEROIN_COT_SYSTEM_PROMPT},
        {"role": "user", "content": question_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = mdl.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    print(reply.replace("\n", " "))
    return parse_cot_answer(reply)

def run_experiment_heroin(model_id, base_df):
    """Run all 27 Kirby trials with heroin user persona."""
    tokenizer, mdl = load_model(model_id)
    df = base_df.copy()
    responses = []
    for _, trial in df.iterrows():
        q = f"Would you prefer ${trial.sir} today, or ${trial.ldr} in {trial.delay} days?"
        ans = ask_question_heroin(q, tokenizer, mdl)
        responses.append(ans)
        print(f"Q{int(trial.order):2d}: SIR=${trial.sir}, LDR=${trial.ldr}, "
              f"delay={trial.delay}d, k_indiff={trial.k_indiff:.4f} => {ans}")
    df["response"] = responses
    df["chose_delayed"] = df["response"].apply(lambda r: r == "delayed")
    df["magnitude"] = df["ldr"].apply(magnitude)
    del mdl, tokenizer
    torch.cuda.empty_cache()
    return df

def run_experiment_heroin_cot(model_id, base_df):
    """Run all 27 Kirby trials with heroin user persona + chain-of-thought."""
    tokenizer, mdl = load_model(model_id)
    df = base_df.copy()
    responses, reasoning = [], []
    for _, trial in df.iterrows():
        q = f"Would you prefer ${trial.sir} today, or ${trial.ldr} in {trial.delay} days?"
        ans, full_reply = ask_question_heroin_cot(q, tokenizer, mdl)
        responses.append(ans)
        reasoning.append(full_reply)
        print(f"  => Q{int(trial.order):2d}: k_indiff={trial.k_indiff:.4f} => {ans}\n")
    df["response"] = responses
    df["reasoning"] = reasoning
    df["chose_delayed"] = df["response"].apply(lambda r: r == "delayed")
    df["magnitude"] = df["ldr"].apply(magnitude)
    del mdl, tokenizer
    torch.cuda.empty_cache()
    return df

### Experiment 3a: Qwen3-4B — Heroin User Persona (no CoT)

In [21]:
df_4b_heroin = run_experiment_heroin("Qwen/Qwen3-4B", trials_df)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B
now | Q 1: SIR=$54.0, LDR=$55.0, delay=117.0d, k_indiff=0.0002 => now
now | Q 2: SIR=$55.0, LDR=$75.0, delay=61.0d, k_indiff=0.0060 => now
now | Q 3: SIR=$19.0, LDR=$25.0, delay=53.0d, k_indiff=0.0060 => now
now | Q 4: SIR=$31.0, LDR=$85.0, delay=7.0d, k_indiff=0.2500 => now
later | Q 5: SIR=$14.0, LDR=$25.0, delay=19.0d, k_indiff=0.0410 => delayed
later | Q 6: SIR=$47.0, LDR=$50.0, delay=160.0d, k_indiff=0.0004 => delayed
now | Q 7: SIR=$15.0, LDR=$35.0, delay=13.0d, k_indiff=0.1000 => now
later | Q 8: SIR=$25.0, LDR=$60.0, delay=14.0d, k_indiff=0.1000 => delayed
now | Q 9: SIR=$78.0, LDR=$80.0, delay=162.0d, k_indiff=0.0002 => now
now | Q10: SIR=$40.0, LDR=$55.0, delay=62.0d, k_indiff=0.0060 => now
now | Q11: SIR=$11.0, LDR=$30.0, delay=7.0d, k_indiff=0.2500 => now
later | Q12: SIR=$67.0, LDR=$75.0, delay=119.0d, k_indiff=0.0010 => delayed
now | Q13: SIR=$34.0, LDR=$35.0, delay=186.0d, k_indiff=0.0002 => now
now | Q14: SIR=$27.0, LDR=$50.0, delay=21.0d, k_indiff=

In [22]:
result_4b_heroin = show_results("Qwen/Qwen3-4B (Heroin Persona)", df_4b_heroin)

print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_4b_heroin.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.250000
Consistency: 19/27 (70.4%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.250000  (consistency 78%)
  medium (LDR $50-$60):  k = 0.126522  (consistency 67%)
   large (LDR $75-$85):  k = 0.126522  (consistency 67%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,delayed
4,20,28,30,179,0.00040,small,delayed
5,17,80,85,157,0.00040,large,now
6,12,67,75,119,0.00100,large,delayed
7,24,54,60,111,0.00100,medium,delayed
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,delayed


### Experiment 3b: Qwen3-4B — Heroin User Persona (CoT)

In [23]:
df_4b_heroin_cot = run_experiment_heroin_cot("Qwen/Qwen3-4B", trials_df)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B
I would prefer $54.0 today.   NOW
  => Q 1: k_indiff=0.0002 => now

I would prefer $55.0 today.   NOW
  => Q 2: k_indiff=0.0060 => now

I would prefer $19.0 today.   NOW
  => Q 3: k_indiff=0.0060 => now

I would prefer $31.0 today because I need the money now for immediate needs, even though $85.0 is more in the future.   NOW
  => Q 4: k_indiff=0.2500 => now

I would prefer $14.0 today.   NOW
  => Q 5: k_indiff=0.0410 => now

I would prefer $47.0 today.   NOW
  => Q 6: k_indiff=0.0004 => now

I would prefer $15.0 today.   NOW
  => Q 7: k_indiff=0.1000 => now

I would prefer $25.0 today.   NOW
  => Q 8: k_indiff=0.1000 => now

I would prefer $78.0 today.   NOW
  => Q 9: k_indiff=0.0002 => now

I would prefer $40.0 today.   NOW
  => Q10: k_indiff=0.0060 => now

I would prefer $11.0 today.   NOW
  => Q11: k_indiff=0.2500 => now

I would prefer $67.0 today.   NOW
  => Q12: k_indiff=0.0010 => now

I would prefer $34.0 today.   NOW
  => Q13: k_indiff=0.0002 => now

I wou

In [24]:
result_4b_heroin_cot = show_results("Qwen/Qwen3-4B (Heroin Persona, CoT)", df_4b_heroin_cot)

print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_4b_heroin_cot.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.250000
Consistency: 26/27 (96.3%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.250000  (consistency 100%)
  medium (LDR $50-$60):  k = 0.250000  (consistency 89%)
   large (LDR $75-$85):  k = 0.250000  (consistency 100%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,now
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,now
6,12,67,75,119,0.00100,large,now
7,24,54,60,111,0.00100,medium,now
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,now


### Experiment 3c: Qwen3-8B — Heroin User Persona (no CoT)

In [25]:
df_8b_heroin = run_experiment_heroin("Qwen/Qwen3-8B", trials_df)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded Qwen/Qwen3-8B
now | Q 1: SIR=$54.0, LDR=$55.0, delay=117.0d, k_indiff=0.0002 => now
later | Q 2: SIR=$55.0, LDR=$75.0, delay=61.0d, k_indiff=0.0060 => delayed
later | Q 3: SIR=$19.0, LDR=$25.0, delay=53.0d, k_indiff=0.0060 => delayed
later | Q 4: SIR=$31.0, LDR=$85.0, delay=7.0d, k_indiff=0.2500 => delayed
later | Q 5: SIR=$14.0, LDR=$25.0, delay=19.0d, k_indiff=0.0410 => delayed
now | Q 6: SIR=$47.0, LDR=$50.0, delay=160.0d, k_indiff=0.0004 => now
now | Q 7: SIR=$15.0, LDR=$35.0, delay=13.0d, k_indiff=0.1000 => now
later | Q 8: SIR=$25.0, LDR=$60.0, delay=14.0d, k_indiff=0.1000 => delayed
now | Q 9: SIR=$78.0, LDR=$80.0, delay=162.0d, k_indiff=0.0002 => now
later | Q10: SIR=$40.0, LDR=$55.0, delay=62.0d, k_indiff=0.0060 => delayed
now | Q11: SIR=$11.0, LDR=$30.0, delay=7.0d, k_indiff=0.2500 => now
later | Q12: SIR=$67.0, LDR=$75.0, delay=119.0d, k_indiff=0.0010 => delayed
later | Q13: SIR=$34.0, LDR=$35.0, delay=186.0d, k_indiff=0.0002 => delayed
later | Q14: SIR=$27.0, LDR=$50

In [26]:
result_8b_heroin = show_results("Qwen/Qwen3-8B (Heroin Persona)", df_8b_heroin)

print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_8b_heroin.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.000632
Consistency: 21/27 (77.8%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.003873  (consistency 67%)
  medium (LDR $50-$60):  k = 0.000632  (consistency 89%)
   large (LDR $75-$85):  k = 0.000632  (consistency 100%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,now
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,delayed
3,6,47,50,160,0.00040,medium,now
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,now
6,12,67,75,119,0.00100,large,delayed
7,24,54,60,111,0.00100,medium,delayed
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,delayed


### Experiment 3d: Qwen3-8B — Heroin User Persona (CoT)

In [27]:
df_8b_heroin_cot = run_experiment_heroin_cot("Qwen/Qwen3-8B", trials_df)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded Qwen/Qwen3-8B
The money is very close in amount, but getting it now would allow me to use it immediately, which could be tempting. However, I'm trying to stay focused on my recovery and long-term goals. I need to think about the bigger picture.    LATER
  => Q 1: k_indiff=0.0002 => delayed

The money is guaranteed, so I don't have to worry about losing it. I need the money now to pay for my treatment and daily expenses. I can't wait 61 days.   NOW
  => Q 2: k_indiff=0.0060 => now

I know that getting money sooner can help with immediate needs, but I also know that waiting can sometimes mean more stability. However, with my current situation, having the money now could help me avoid stress and potential relapse.    NOW
  => Q 3: k_indiff=0.0060 => now

The money is guaranteed, so I don't have to worry about losing it. I need the money now to pay for my treatment and daily expenses. I can't wait 7 days.   NOW
  => Q 4: k_indiff=0.2500 => now

The money is guaranteed, so I don't ha

In [28]:
result_8b_heroin_cot = show_results("Qwen/Qwen3-8B (Heroin Persona, CoT)", df_8b_heroin_cot)

print(f"\n\nAll responses (sorted by k at indifference):\n")
display_df = df_8b_heroin_cot.sort_values("k_indiff")[
    ["order", "sir", "ldr", "delay", "k_indiff", "magnitude", "response"]
].reset_index(drop=True)
display_df.columns = ["Q#", "SIR ($)", "LDR ($)", "Delay (days)",
                       "k at indiff.", "Magnitude", "LLM choice"]
display_df

Estimated discount rate  k = 0.250000
Consistency: 19/27 (70.4%)

Discount rate (k) by reward magnitude:

   small (LDR $25-$35):  k = 0.022316  (consistency 67%)
  medium (LDR $50-$60):  k = 0.001581  (consistency 67%)
   large (LDR $75-$85):  k = 0.250000  (consistency 89%)


All responses (sorted by k at indifference):



,Q#,SIR ($),LDR ($),Delay (days),k at indiff.,Magnitude,LLM choice
0,1,54,55,117,0.00016,medium,delayed
1,9,78,80,162,0.00016,large,now
2,13,34,35,186,0.00016,small,now
3,6,47,50,160,0.00040,medium,now
4,20,28,30,179,0.00040,small,now
5,17,80,85,157,0.00040,large,now
6,12,67,75,119,0.00100,large,now
7,24,54,60,111,0.00100,medium,now
8,26,22,25,136,0.00100,small,now
9,15,69,85,91,0.00250,large,now


## Full Comparison: Default Persona vs. Heroin Persona vs. Human Benchmarks

In [29]:
# ---------- Full comparison: all experiments ----------
print(f"  {'Group':<40s}  {'k':>10s}  {'Consistency':>12s}")
print(f"  {'-'*40}  {'-'*10}  {'-'*12}")

# Default persona
print("  — Default persona (35yo, stable job) —")
if 'result_4b' in dir():
    print(f"  {'  Qwen3-4B':<40s}  {result_4b['k']:>10.6f}  {result_4b['consistency']:>11.1%}")
if 'result_4b_cot' in dir():
    print(f"  {'  Qwen3-4B (CoT)':<40s}  {result_4b_cot['k']:>10.6f}  {result_4b_cot['consistency']:>11.1%}")
if 'result_8b' in dir():
    print(f"  {'  Qwen3-8B':<40s}  {result_8b['k']:>10.6f}  {result_8b['consistency']:>11.1%}")
if 'result_8b_cot' in dir():
    print(f"  {'  Qwen3-8B (CoT)':<40s}  {result_8b_cot['k']:>10.6f}  {result_8b_cot['consistency']:>11.1%}")

# Heroin persona
print("  — Heroin user persona (Kirby study) —")
if 'result_4b_heroin' in dir():
    print(f"  {'  Qwen3-4B':<40s}  {result_4b_heroin['k']:>10.6f}  {result_4b_heroin['consistency']:>11.1%}")
if 'result_4b_heroin_cot' in dir():
    print(f"  {'  Qwen3-4B (CoT)':<40s}  {result_4b_heroin_cot['k']:>10.6f}  {result_4b_heroin_cot['consistency']:>11.1%}")
if 'result_8b_heroin' in dir():
    print(f"  {'  Qwen3-8B':<40s}  {result_8b_heroin['k']:>10.6f}  {result_8b_heroin['consistency']:>11.1%}")
if 'result_8b_heroin_cot' in dir():
    print(f"  {'  Qwen3-8B (CoT)':<40s}  {result_8b_heroin_cot['k']:>10.6f}  {result_8b_heroin_cot['consistency']:>11.1%}")

# API-based LLMs
print("  — API-based LLMs (default persona) —")
for name, result in human_results.items():
    print(f"  {'  ' + name:<40s}  {result['k']:>10.6f}  {result['consistency']:>11.1%}")

# Human benchmarks
print("  — Human benchmarks (Kirby 1999) —")
print(f"  {'  Non-drug controls':<40s}  {'0.013':>10s}  {'96%':>12s}")
print(f"  {'  Heroin patients':<40s}  {'0.025':>10s}  {'94%':>12s}")

  Group                                              k   Consistency
  ----------------------------------------  ----------  ------------
  — Default persona (35yo, stable job) —
    Qwen3-4B                                  0.025612        77.8%
    Qwen3-4B (CoT)                            0.025612        81.5%
    Qwen3-8B                                  0.009960        74.1%
    Qwen3-8B (CoT)                            0.000632        77.8%
  — Heroin user persona (Kirby study) —
    Qwen3-4B                                  0.250000        70.4%
    Qwen3-4B (CoT)                            0.250000        96.3%
    Qwen3-8B                                  0.000632        77.8%
    Qwen3-8B (CoT)                            0.250000        70.4%
  — API-based LLMs (default persona) —
    Gemini                                    0.001581        96.3%
    Claude                                    0.001581        96.3%
  — Human benchmarks (Kirby 1999) —
    Non-drug controls     